# Exercise 9: Strict-Mode Structured Output

Use `text.format` with a JSON schema in **strict mode** to guarantee the model returns
well-typed, schema-compliant JSON every time.

Scenario: generate a **Customer Health Assessment** from messy account signals.

In [1]:
import json

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

## Define the JSON schema

All fields are `required`, `additionalProperties` is `False`, and enums constrain categorical values.

In [2]:
schema = {
    "type": "json_schema",
    "name": "customer_health_assessment",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "customer_name": {"type": "string"},
            "assessment_date": {"type": "string", "description": "ISO date format"},
            "health_score": {
                "type": "number",
                "description": "0-100 score based on overall health signals",
            },
            "risk_level": {
                "type": "string",
                "enum": ["healthy", "monitor", "at_risk", "critical"],
            },
            "usage_trend": {
                "type": "string",
                "enum": ["increasing", "stable", "declining"],
            },
            "key_signals": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "signal": {"type": "string"},
                        "sentiment": {
                            "type": "string",
                            "enum": ["positive", "neutral", "negative"],
                        },
                        "detail": {"type": "string"},
                    },
                    "required": ["signal", "sentiment", "detail"],
                    "additionalProperties": False,
                },
            },
            "recommended_actions": {
                "type": "array",
                "items": {"type": "string"},
            },
            "next_review_date": {"type": "string"},
        },
        "required": [
            "customer_name",
            "assessment_date",
            "health_score",
            "risk_level",
            "usage_trend",
            "key_signals",
            "recommended_actions",
            "next_review_date",
        ],
        "additionalProperties": False,
    },
}

print("Schema defined:", schema["name"])

Schema defined: customer_health_assessment


## Generate the structured assessment

In [3]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input="""Based on this account context, generate a customer health assessment:

Customer: Meridian Financial Services
- Contract: $480K/year, renewed 6 months ago
- API usage down 23% month-over-month for the last 3 months
- 2 P1 support tickets in the last 30 days (auth failures on their SSO integration)
- Champion contact (VP of Engineering) left the company 2 weeks ago
- Last QBR was 4 months ago, they cancelled the most recent one
- They recently started evaluating a competitor (found in their public RFP)
""",
    text={"format": schema},
)

result = json.loads(response.output_text)
print("=== Customer Health Assessment (Structured Output) ===\n")
print(json.dumps(result, indent=2))

=== Customer Health Assessment (Structured Output) ===

{
  "customer_name": "Meridian Financial Services",
  "assessment_date": "2024-06-12",
  "health_score": 55,
  "risk_level": "at_risk",
  "usage_trend": "declining",
  "key_signals": [
    {
      "signal": "API usage",
      "sentiment": "negative",
      "detail": "Down 23% month-over-month for the last 3 months indicating decreased engagement."
    },
    {
      "signal": "Support tickets",
      "sentiment": "negative",
      "detail": "2 P1 tickets in last 30 days due to authentication failures affecting SSO integration."
    },
    {
      "signal": "Champion contact",
      "sentiment": "negative",
      "detail": "VP of Engineering and champion contact left the company 2 weeks ago."
    },
    {
      "signal": "QBR engagement",
      "sentiment": "negative",
      "detail": "Cancelled most recent Quarterly Business Review, last meeting was 4 months ago."
    },
    {
      "signal": "Competitive evaluation",
      "senti

## Schema compliance check

In [4]:
print("=== Schema Compliance Check ===")
expected_fields = schema["schema"]["required"]
actual_fields = list(result.keys())
print(f"Expected fields: {expected_fields}")
print(f"Actual fields:   {actual_fields}")
print(f"All required fields present: {all(f in result for f in expected_fields)}")
print(f"Risk level valid enum: {result['risk_level'] in ['healthy', 'monitor', 'at_risk', 'critical']}")
print(f"Usage trend valid enum: {result['usage_trend'] in ['increasing', 'stable', 'declining']}")

for sig in result["key_signals"]:
    assert set(sig.keys()) == {"signal", "sentiment", "detail"}, f"Bad signal: {sig}"
    assert sig["sentiment"] in ["positive", "neutral", "negative"], f"Bad sentiment: {sig['sentiment']}"
print(f"All {len(result['key_signals'])} signals have valid structure and sentiment enums")

print(f"\nTokens: {response.usage.input_tokens} in, {response.usage.output_tokens} out")

=== Schema Compliance Check ===
Expected fields: ['customer_name', 'assessment_date', 'health_score', 'risk_level', 'usage_trend', 'key_signals', 'recommended_actions', 'next_review_date']
Actual fields:   ['customer_name', 'assessment_date', 'health_score', 'risk_level', 'usage_trend', 'key_signals', 'recommended_actions', 'next_review_date']
All required fields present: True
Risk level valid enum: True
Usage trend valid enum: True
All 5 signals have valid structure and sentiment enums

Tokens: 295 in, 272 out


---

**Interview one-liner:** Strict-mode structured output with `text.format` guarantees the LLM response parses as valid JSON matching your schema -- every field, every enum, every time -- eliminating fragile regex post-processing.